# InfographicVQA QA Benchmark Evaluation

Comprehensive evaluation of OCR and Question Answering (QA) performance on the InfographicVQA mini dataset with **unique pre-extracted OCR analysis**.

## Pipeline Overview

**QA Phases (11 total):**
- **QA1a-c**: OCR Pipeline (Extract text via OCR → Answer QA based on extracted text)
  - QA1a: Simple prompts
  - QA1b: Detailed prompts with context
  - QA1c: Chain-of-Thought prompting
  
- **QA2a-c**: VLM Parse Pipeline (Parse document with VLM → Answer QA based on parsed output)
  - QA2a: Simple prompts
  - QA2b: Detailed prompts
  - QA2c: Chain-of-Thought prompting
  
- **QA3a-b**: Direct VQA (VLM answers directly from image)
  - QA3a: Simple prompts
  - QA3b: Chain-of-Thought prompting

- **QA4a-c**: Pre-extracted OCR Pipeline ⭐ UNIQUE to InfographicVQA
  - Uses AWS Textract OCR from metadata
  - QA4a: Simple prompts
  - QA4b: Detailed prompts
  - QA4c: Chain-of-Thought prompting

**Models Evaluated:**
- OCR (QA1): `azure_intelligence`, `mistral_document_ai`
- VLM (QA2): `gpt-5-mini`, `gpt-5-nano`
- Direct VQA (QA3): `gpt-5-mini`, `gpt-5-nano`
- Pre-extracted OCR (QA4): AWS Textract (metadata)
- QA: `gpt-5-mini`

## 1. Setup and Data Loading

In [ ]:
%pip install pandas numpy matplotlib seaborn scipy scikit-learn -q

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple
import warnings
import sys
warnings.filterwarnings('ignore')

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

# Import our evaluation metrics
from evaluation_metrics import (
    calculate_cer, calculate_wer, compute_anls, compute_exact_match,
    compute_substring_match, compute_cosine_similarity, compute_all_vqa_metrics
)

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Setup plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ Libraries and evaluation metrics loaded successfully!")

In [ ]:
DATASET = "InfographicVQA_mini"
RESULTS_DIR = Path("../results") / DATASET

# Define phases and their descriptions
PHASES = {
    "QA1a": "OCR Pipeline - Simple Prompts (azure_intelligence)",
    "QA1b": "OCR Pipeline - Detailed Prompts (azure_intelligence)",
    "QA1c": "OCR Pipeline - Chain-of-Thought (azure_intelligence)",
    "QA2a": "VLM Parse Pipeline - Simple (gpt-5-mini/nano)",
    "QA2b": "VLM Parse Pipeline - Detailed (gpt-5-mini/nano)",
    "QA2c": "VLM Parse Pipeline - Chain-of-Thought (gpt-5-mini/nano)",
    "QA3a": "Direct VQA - Simple (gpt-5-mini/nano)",
    "QA3b": "Direct VQA - Chain-of-Thought (gpt-5-mini/nano)",
    "QA4a": "Pre-extracted OCR Pipeline - Simple (AWS Textract) ⭐",
    "QA4b": "Pre-extracted OCR Pipeline - Detailed (AWS Textract) ⭐",
    "QA4c": "Pre-extracted OCR Pipeline - Chain-of-Thought (AWS Textract) ⭐",
}

# Phase groupings (NOTE: QA4 is unique to InfographicVQA)
APPROACH_GROUPS = {
    "OCR Pipeline": ["QA1a", "QA1b", "QA1c"],
    "VLM Parse Pipeline": ["QA2a", "QA2b", "QA2c"],
    "Direct VQA": ["QA3a", "QA3b"],
    "Pre-extracted OCR Pipeline": ["QA4a", "QA4b", "QA4c"],  # UNIQUE to InfographicVQA
}

print(f"Dataset: {DATASET}")
print(f"Results Directory: {RESULTS_DIR}")
print(f"Phases to analyze: {len(PHASES)}")
for phase, desc in PHASES.items():
    print(f"  - {phase}: {desc}")

In [ ]:
# Load execution summary
summary_file = RESULTS_DIR / "execution_summary.json"
with open(summary_file) as f:
    execution_summary = json.load(f)

print(f"Loaded execution summary from {summary_file}")
print(f"Start time: {execution_summary.get('start_time', 'N/A')}")
print(f"End time: {execution_summary.get('end_time', 'N/A')}")
print(f"Total runtime: {execution_summary.get('total_time_seconds', 0) / 3600:.2f} hours")
print(f"\nMetrics summary available: {'metrics_summary' in execution_summary}")

In [ ]:
# Load all result CSV files for each phase
all_results = {}
for phase in PHASES.keys():
    phase_dir = RESULTS_DIR / phase
    if not phase_dir.exists():
        print(f"⚠ Phase {phase} directory not found at {phase_dir}")
        continue
    
    phase_results = []
    for csv_file in phase_dir.glob("*_results.csv"):
        df = pd.read_csv(csv_file)
        phase_results.append(df)
    
    if phase_results:
        all_results[phase] = pd.concat(phase_results, ignore_index=True)
        print(f"✓ Phase {phase}: {len(all_results[phase])} samples")
    else:
        print(f"⚠ No results found for phase {phase}")

print(f"\n✓ Loaded results for {len(all_results)} phases")

## 2. OCR Evaluation Metrics

In [ ]:
def analyze_extracted_text(df: pd.DataFrame) -> Dict:
    """Analyze OCR extraction quality from extracted_text column."""
    results = {
        'total_samples': len(df),
        'samples_with_text': (df['extracted_text'].notna() & (df['extracted_text'] != '')).sum(),
        'empty_extractions': (df['extracted_text'].isna() | (df['extracted_text'] == '')).sum(),
        'avg_text_length': df['extracted_text'].fillna('').apply(lambda x: len(str(x))).mean(),
        'median_text_length': df['extracted_text'].fillna('').apply(lambda x: len(str(x))).median(),
    }
    results['extraction_success_rate'] = results['samples_with_text'] / results['total_samples']
    return results

# Analyze OCR extraction for QA1 phases (OCR Pipeline)
print("OCR Extraction Analysis (QA1 phases - OCR Pipeline)")
print("=" * 70)
ocr_phases = ["QA1a", "QA1b", "QA1c"]
ocr_analysis = {}

for phase in ocr_phases:
    if phase not in all_results:
        continue
    df = all_results[phase]
    analysis = analyze_extracted_text(df)
    ocr_analysis[phase] = analysis
    
    print(f"\n{phase}:")
    print(f"  Total samples: {analysis['total_samples']}")
    print(f"  Successful extractions: {analysis['samples_with_text']} ({analysis['extraction_success_rate']*100:.1f}%)")
    print(f"  Avg text length: {analysis['avg_text_length']:.0f} chars")
    print(f"  Median text length: {analysis['median_text_length']:.0f} chars")

# Analyze pre-extracted OCR (QA4 phases) - unique to InfographicVQA
print("\n\nPre-extracted OCR Analysis (QA4 phases - AWS Textract from metadata) ⭐")
print("=" * 70)
preextracted_ocr_phases = ["QA4a", "QA4b", "QA4c"]
for phase in preextracted_ocr_phases:
    if phase not in all_results:
        continue
    df = all_results[phase]
    analysis = analyze_extracted_text(df)
    
    print(f"\n{phase}:")
    print(f"  Total samples: {analysis['total_samples']}")
    print(f"  Successful extractions: {analysis['samples_with_text']} ({analysis['extraction_success_rate']*100:.1f}%)")
    print(f"  Avg text length: {analysis['avg_text_length']:.0f} chars")
    print(f"  Median text length: {analysis['median_text_length']:.0f} chars")

## 3. OCR Performance Analysis

In [ ]:
# Analyze which OCR models appear in each QA1 phase
print("OCR Models in QA1 Phases:")
print("=" * 70)
for phase in ocr_phases:
    if phase not in all_results:
        continue
    df = all_results[phase]
    models = df['parsing_model'].unique()
    print(f"\n{phase}:")
    for model in sorted(models):
        count = (df['parsing_model'] == model).sum()
        print(f"  {model}: {count} samples")

# Note about QA4: Pre-extracted OCR comes from metadata (AWS Textract)
print("\n\nQA4 Phases (Pre-extracted OCR) ⭐:")
print("=" * 70)
print("QA4a-c: Pre-extracted OCR from AWS Textract metadata")
print("  - Not model-dependent (uses fixed metadata)")
print("  - Serves as upper bound for 'perfect' OCR scenario")

## 4. QA Evaluation Metrics

In [ ]:
# Extract QA metrics from execution summary
print("QA Performance Metrics from Execution Summary")
print("=" * 80)

qa_metrics = []
for phase, desc in PHASES.items():
    if phase not in execution_summary.get('metrics_summary', {}):
        continue
    
    metrics = execution_summary['metrics_summary'][phase]
    qa_metrics.append({
        'Phase': phase,
        'Description': desc,
        'ANLS': metrics.get('anls', 0),
        'Exact Match': metrics.get('exact_match', 0),
        'Total Samples': metrics.get('total_samples', 0),
        'Valid Samples': metrics.get('valid_samples', 0),
        'Error Rate': metrics.get('error_rate', 0),
    })

qa_df = pd.DataFrame(qa_metrics)
print("\nAll Phases:")
print(qa_df.to_string(index=False))

In [ ]:
# Recompute comprehensive metrics from CSV data with all metrics
print("\nRecomputing Comprehensive Metrics (ANLS, EM, CER, WER, Substring Match, Cosine Similarity)")
print("=" * 100)

comprehensive_metrics = []

for phase in PHASES.keys():
    if phase not in all_results:
        continue
    
    df = all_results[phase]
    
    # Skip if no prediction or ground_truth columns
    if 'prediction' not in df.columns or 'ground_truth' not in df.columns:
        print(f"⚠ Phase {phase}: Missing prediction or ground_truth columns")
        continue
    
    metrics_list = []
    
    for idx, row in df.iterrows():
        pred = str(row.get('prediction', ''))
        gt_raw = row.get('ground_truth', '')
        
        # Parse ground truth - could be string or list
        if isinstance(gt_raw, str):
            # Try to parse as JSON list
            try:
                import json
                gt_list = json.loads(gt_raw)
                if not isinstance(gt_list, list):
                    gt_list = [str(gt_raw)]
            except:
                gt_list = [str(gt_raw)]
        else:
            gt_list = [str(gt_raw)]
        
        # Compute all VQA metrics
        vqa_metrics = compute_all_vqa_metrics(pred, gt_list)
        
        # Add CER and WER (using first ground truth as reference)
        if gt_list:
            vqa_metrics['cer'] = calculate_cer(pred, gt_list[0])
            vqa_metrics['wer'] = calculate_wer(pred, gt_list[0])
        else:
            vqa_metrics['cer'] = 1.0
            vqa_metrics['wer'] = 1.0
        
        metrics_list.append(vqa_metrics)
    
    if metrics_list:
        # Aggregate metrics
        avg_metrics = {
            'Phase': phase,
            'ANLS': np.mean([m['anls'] for m in metrics_list]),
            'Exact Match': np.mean([m['exact_match'] for m in metrics_list]),
            'Substring Match': np.mean([m['substring_match'] for m in metrics_list]),
            'Cosine Similarity': np.mean([m['cosine_similarity'] for m in metrics_list]),
            'CER': np.mean([m['cer'] for m in metrics_list]),
            'WER': np.mean([m['wer'] for m in metrics_list]),
            'Samples': len(metrics_list)
        }
        comprehensive_metrics.append(avg_metrics)
        print(f"\n{phase}:")
        print(f"  ANLS:              {avg_metrics['ANLS']:.4f}")
        print(f"  Exact Match:       {avg_metrics['Exact Match']:.4f}")
        print(f"  Substring Match:   {avg_metrics['Substring Match']:.4f}")
        print(f"  Cosine Similarity: {avg_metrics['Cosine Similarity']:.4f}")
        print(f"  CER:               {avg_metrics['CER']:.4f}")
        print(f"  WER:               {avg_metrics['WER']:.4f}")
        print(f"  Samples:           {avg_metrics['Samples']}")

# Create comprehensive DataFrame
comprehensive_df = pd.DataFrame(comprehensive_metrics)
print("\n" + "=" * 100)
print("Comprehensive Metrics Summary:")
print(comprehensive_df.to_string(index=False))

In [ ]:
# Visualize all metrics comparison
if len(comprehensive_df) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Comprehensive Evaluation Metrics Across All Phases', fontsize=16, fontweight='bold')
    
    metrics_to_plot = [
        ('ANLS', 'Higher is Better'),
        ('Exact Match', 'Higher is Better'),
        ('Substring Match', 'Higher is Better'),
        ('Cosine Similarity', 'Higher is Better'),
        ('CER', 'Lower is Better'),
        ('WER', 'Lower is Better')
    ]
    
    for idx, (metric, direction) in enumerate(metrics_to_plot):
        ax = axes[idx // 3, idx % 3]
        
        if metric in comprehensive_df.columns:
            data = comprehensive_df.sort_values(metric, ascending=(direction == 'Lower is Better'))
            
            colors = plt.cm.RdYlGn(data[metric] if direction == 'Higher is Better' else 1 - data[metric])
            bars = ax.barh(range(len(data)), data[metric], color=colors)
            
            ax.set_yticks(range(len(data)))
            ax.set_yticklabels(data['Phase'], fontsize=8)
            ax.set_xlabel(f'{metric} Score', fontsize=10)
            ax.set_title(f'{metric}\n({direction})', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='x')
            
            # Add value labels on bars
            for i, (bar, val) in enumerate(zip(bars, data[metric])):
                ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                       f'{val:.3f}', va='center', fontsize=8)
    
    plt.tight_layout()
    plt.show()
else:
    print("No comprehensive metrics available to visualize")

In [ ]:
# Group metrics by approach
print("\n\nPerformance by Approach:")
print("=" * 80)

approach_metrics = []
for approach, phases in APPROACH_GROUPS.items():
    valid_phases = [p for p in phases if p in execution_summary.get('metrics_summary', {})]
    if not valid_phases:
        continue
    
    anls_scores = [execution_summary['metrics_summary'][p].get('anls', 0) for p in valid_phases]
    em_scores = [execution_summary['metrics_summary'][p].get('exact_match', 0) for p in valid_phases]
    
    approach_metrics.append({
        'Approach': approach,
        'Avg ANLS': np.mean(anls_scores),
        'Std ANLS': np.std(anls_scores),
        'Avg EM': np.mean(em_scores),
        'Std EM': np.std(em_scores),
        'Num Phases': len(valid_phases),
    })

approach_df = pd.DataFrame(approach_metrics)
print(approach_df.to_string(index=False))

# Store for later visualization
approach_metrics_data = approach_df

## 5. QA Performance Analysis

In [ ]:
# Analyze prompt effectiveness within OCR Pipeline
print("\n\nPrompt Variation Analysis within OCR Pipeline (QA1 phases):")
print("=" * 80)
ocr_qa_analysis = []
for phase in ["QA1a", "QA1b", "QA1c"]:
    if phase not in execution_summary.get('metrics_summary', {}):
        continue
    
    metrics = execution_summary['metrics_summary'][phase]
    ocr_qa_analysis.append({
        'Phase': phase,
        'Prompt Style': phase.split('1')[1].upper(),
        'ANLS': metrics.get('anls', 0),
        'EM': metrics.get('exact_match', 0),
    })

ocr_qa_df = pd.DataFrame(ocr_qa_analysis)
print(ocr_qa_df.to_string(index=False))
if len(ocr_qa_df) > 0:
    print(f"\nBest OCR prompt variant: {ocr_qa_df.loc[ocr_qa_df['ANLS'].idxmax(), 'Phase']} (ANLS: {ocr_qa_df['ANLS'].max():.4f})")

In [ ]:
# Analyze prompt effectiveness within VLM Parse Pipeline
print("\n\nPrompt Variation Analysis within VLM Parse Pipeline (QA2 phases):")
print("=" * 80)
vlm_qa_analysis = []
for phase in ["QA2a", "QA2b", "QA2c"]:
    if phase not in execution_summary.get('metrics_summary', {}):
        continue
    
    metrics = execution_summary['metrics_summary'][phase]
    vlm_qa_analysis.append({
        'Phase': phase,
        'Prompt Style': phase.split('2')[1].upper(),
        'ANLS': metrics.get('anls', 0),
        'EM': metrics.get('exact_match', 0),
    })

vlm_qa_df = pd.DataFrame(vlm_qa_analysis)
print(vlm_qa_df.to_string(index=False))
if len(vlm_qa_df) > 0:
    print(f"\nBest VLM Parse prompt variant: {vlm_qa_df.loc[vlm_qa_df['ANLS'].idxmax(), 'Phase']} (ANLS: {vlm_qa_df['ANLS'].max():.4f})")

In [ ]:
# Analyze Direct VQA variants
print("\n\nDirect VQA Analysis (QA3 phases):")
print("=" * 80)
direct_qa_analysis = []
for phase in ["QA3a", "QA3b"]:
    if phase not in execution_summary.get('metrics_summary', {}):
        continue
    
    metrics = execution_summary['metrics_summary'][phase]
    direct_qa_analysis.append({
        'Phase': phase,
        'Prompt Style': phase.split('3')[1].upper(),
        'ANLS': metrics.get('anls', 0),
        'EM': metrics.get('exact_match', 0),
    })

direct_qa_df = pd.DataFrame(direct_qa_analysis)
print(direct_qa_df.to_string(index=False))
if len(direct_qa_df) > 1:
    print(f"\nBest Direct VQA variant: {direct_qa_df.loc[direct_qa_df['ANLS'].idxmax(), 'Phase']} (ANLS: {direct_qa_df['ANLS'].max():.4f})")

In [ ]:
# Analyze Pre-extracted OCR Pipeline variants - UNIQUE TO INFOGRAPHICVQA ⭐
print("\n\nPre-extracted OCR Pipeline Analysis (QA4 phases) ⭐ UNIQUE TO INFOGRAPHICVQA:")
print("=" * 80)
preextracted_qa_analysis = []
for phase in ["QA4a", "QA4b", "QA4c"]:
    if phase not in execution_summary.get('metrics_summary', {}):
        continue
    
    metrics = execution_summary['metrics_summary'][phase]
    preextracted_qa_analysis.append({
        'Phase': phase,
        'Prompt Style': phase.split('4')[1].upper(),
        'ANLS': metrics.get('anls', 0),
        'EM': metrics.get('exact_match', 0),
    })

preextracted_qa_df = pd.DataFrame(preextracted_qa_analysis)
if len(preextracted_qa_df) > 0:
    print(preextracted_qa_df.to_string(index=False))
    print(f"\nBest Pre-extracted OCR prompt variant: {preextracted_qa_df.loc[preextracted_qa_df['ANLS'].idxmax(), 'Phase']} (ANLS: {preextracted_qa_df['ANLS'].max():.4f})")
    print("\n💡 INSIGHT: Pre-extracted OCR from AWS Textract represents the 'upper bound'")
    print("   for OCR-based QA when using perfect/high-quality OCR extraction.")

## 6. Cross-Dataset & Cross-Approach Comparison

In [ ]:
print("APPROACH COMPARISON - Key Metrics")
print("=" * 80)

# Get best phase from each approach
best_by_approach = {}
for approach, phases in APPROACH_GROUPS.items():
    valid_phases = [p for p in phases if p in execution_summary.get('metrics_summary', {})]
    if valid_phases:
        anls_scores = {p: execution_summary['metrics_summary'][p].get('anls', 0) for p in valid_phases}
        best_phase = max(anls_scores, key=anls_scores.get)
        best_by_approach[approach] = {
            'best_phase': best_phase,
            'anls': anls_scores[best_phase],
            'em': execution_summary['metrics_summary'][best_phase].get('exact_match', 0),
        }

for approach, data in best_by_approach.items():
    marker = " ⭐" if "Pre-extracted" in approach else ""
    print(f"\n{approach}{marker}")
    print(f"  Best phase: {data['best_phase']}")
    print(f"  ANLS: {data['anls']:.4f}")
    print(f"  Exact Match: {data['em']:.4f}")

# Overall best
if best_by_approach:
    best_overall = max(best_by_approach.items(), key=lambda x: x[1]['anls'])
    print(f"\n{'=' * 80}")
    print(f"OVERALL BEST: {best_overall[0]} (Phase: {best_overall[1]['best_phase']})")
    print(f"ANLS: {best_overall[1]['anls']:.4f} | EM: {best_overall[1]['em']:.4f}")

## 7. Visualizations & Summary

In [ ]:
# Create phase performance visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ANLS Scores
ax1 = axes[0]
phases = [p for p in PHASES.keys() if p in execution_summary.get('metrics_summary', {})]
anls_scores = [execution_summary['metrics_summary'][p].get('anls', 0) for p in phases]
colors = ['#FF6B6B' if p.startswith('QA1') else '#4ECDC4' if p.startswith('QA2') else '#95E1D3' if p.startswith('QA3') else '#FFD93D' for p in phases]

ax1.bar(phases, anls_scores, color=colors, alpha=0.8, edgecolor='black')
ax1.set_ylabel('ANLS Score', fontsize=12, fontweight='bold')
ax1.set_title('ANLS Scores Across All Phases (InfographicVQA)', fontsize=14, fontweight='bold')
ax1.set_ylim([0, 1])
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (phase, score) in enumerate(zip(phases, anls_scores)):
    ax1.text(i, score + 0.02, f'{score:.3f}', ha='center', va='bottom', fontsize=8)

# Exact Match Scores
ax2 = axes[1]
em_scores = [execution_summary['metrics_summary'][p].get('exact_match', 0) for p in phases]

ax2.bar(phases, em_scores, color=colors, alpha=0.8, edgecolor='black')
ax2.set_ylabel('Exact Match Rate', fontsize=12, fontweight='bold')
ax2.set_title('Exact Match Scores Across All Phases (InfographicVQA)', fontsize=14, fontweight='bold')
ax2.set_ylim([0, 1])
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for i, (phase, score) in enumerate(zip(phases, em_scores)):
    ax2.text(i, score + 0.02, f'{score:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print("\n✓ Phase performance visualization complete")
print("\nColor Legend:")
print("  Red: QA1 (OCR Pipeline)")
print("  Teal: QA2 (VLM Parse Pipeline)")
print("  Green: QA3 (Direct VQA)")
print("  Yellow: QA4 (Pre-extracted OCR Pipeline) ⭐")

In [ ]:
# Approach comparison visualization
fig, ax = plt.subplots(figsize=(13, 6))

approaches = approach_metrics_data['Approach'].values
anls_avg = approach_metrics_data['Avg ANLS'].values
anls_std = approach_metrics_data['Std ANLS'].values

x = np.arange(len(approaches))
width = 0.35

colors_bar = ['#FF6B6B', '#4ECDC4', '#95E1D3', '#FFD93D']
bars = ax.bar(x, anls_avg, width, label='ANLS', color=colors_bar, alpha=0.8, edgecolor='black')
ax.errorbar(x, anls_avg, yerr=anls_std, fmt='none', ecolor='black', capsize=5, capthick=2)

# Add EM scores as secondary
em_avg = approach_metrics_data['Avg EM'].values
ax.bar(x + width, em_avg, width, label='Exact Match', color=[c for c in colors_bar], alpha=0.5, edgecolor='black')

ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Average QA Metrics by Approach (InfographicVQA)', fontsize=14, fontweight='bold')
ax.set_xticks(x + width / 2)
ax.set_xticklabels(approaches, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, (anls, em) in enumerate(zip(anls_avg, em_avg)):
    ax.text(i, anls + 0.03, f'{anls:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i + width, em + 0.03, f'{em:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\n✓ Approach comparison visualization complete")

In [ ]:
# Summary statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS - InfographicVQA Mini Dataset")
print("=" * 80)

all_anls = [execution_summary['metrics_summary'][p].get('anls', 0) 
            for p in PHASES.keys() if p in execution_summary.get('metrics_summary', {})]
all_em = [execution_summary['metrics_summary'][p].get('exact_match', 0) 
          for p in PHASES.keys() if p in execution_summary.get('metrics_summary', {})]

print(f"\nANLS Scores across all {len(all_anls)} phases:")
print(f"  Mean:   {np.mean(all_anls):.4f}")
print(f"  Std:    {np.std(all_anls):.4f}")
print(f"  Min:    {np.min(all_anls):.4f}")
print(f"  Max:    {np.max(all_anls):.4f}")
print(f"  Median: {np.median(all_anls):.4f}")

print(f"\nExact Match Scores across all {len(all_em)} phases:")
print(f"  Mean:   {np.mean(all_em):.4f}")
print(f"  Std:    {np.std(all_em):.4f}")
print(f"  Min:    {np.min(all_em):.4f}")
print(f"  Max:    {np.max(all_em):.4f}")
print(f"  Median: {np.median(all_em):.4f}")

print(f"\nExecution time: {execution_summary.get('total_time_seconds', 0) / 3600:.2f} hours")
print(f"Total samples processed: 500 × {len(PHASES)} phases = {500 * len(PHASES)} evaluations")

# Special metrics for QA4 (Pre-extracted OCR)
qa4_phases = [p for p in ["QA4a", "QA4b", "QA4c"] if p in execution_summary.get('metrics_summary', {})]
if qa4_phases:
    qa4_anls = [execution_summary['metrics_summary'][p].get('anls', 0) for p in qa4_phases]
    print(f"\n⭐ Pre-extracted OCR (QA4) Performance:")
    print(f"  Average ANLS: {np.mean(qa4_anls):.4f}")
    print(f"  Best phase: {max(qa4_phases, key=lambda p: execution_summary['metrics_summary'][p].get('anls', 0))}")